# 05 · Orchestration

So far the agent decided every step. Many systems are better as a **workflow**, where **code** decides
the steps and the model fills in the parts that need language. CrewAI's answer is the **Flow**: plain
Python methods, wired together with decorators, sharing a typed state. Agents and crews run inside its
steps.

**You will learn**

- how to build a Flow: `@start`, `@listen`, `@router` and the flow's state
- when to use a workflow, an agent, or an agent inside a workflow
- how to build a loop whose limit is enforced in code, not in the prompt
- the research agent's design, which the capstone builds on

In [1]:
# Imports and setup shared by the cells below.
from crew_agents.llm import make_llm  # import crew_agents first: it switches CrewAI's telemetry off
from crew_agents.prompts import BACKSTORY, GOAL, ROLE
from crew_agents.tools import shipfast_tools
from crewai import Agent
from crewai.flow.flow import Flow, listen, router, start
from pydantic import BaseModel

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast
from agent_contracts.shipfast.schema import Triage

verify_model()  # stops with the fix if Ollama or the shared model is missing
llm = make_llm()

## A first flow: routing ShipFast messages

A Flow is a class. Its **state** is a Pydantic model every step can read and change (`self.state`).
`@start()` marks the first step; `@listen(step)` runs a method when another finishes; and `@router(step)`
runs after a step and returns a **label**, so only the listeners for that label run next.

ShipFast's first decision is a good fit for code: triage the message (notebook 01), and if it is out of
scope, decline without ever starting the specialist agent. First, the pieces the flow will use: the
triage instructions and the specialist agent from notebook 03.

In [2]:
# The triage instructions: what each intent means, and when order_id should be null.
TRIAGE_PROMPT = ("Classify the customer's message for ShipFast support. intent: order_status, refund_request, "
                 "complaint, or other (anything not about orders, deliveries, refunds or complaints). "
                 "order_id: the order ID such as A1001 if given, else null.")

# The tool-using agent from notebook 03. The flow will start it only for in-scope messages.
specialist = Agent(role=ROLE, goal=GOAL, backstory=BACKSTORY, tools=shipfast_tools(), llm=llm, verbose=False)

Next, the flow's **state**: a Pydantic model whose fields every step can read and change. Every field has a default, so the flow can start with just the message.

In [3]:
class Support(BaseModel):
    message: str = ""  # the customer's message: the input
    intent: str = ""  # set by triage
    order_id: str | None = None  # set by triage
    reply: str = ""  # set by a decline or resolve step: the output

Now the flow itself. Read it top to bottom: `triage` starts it, `route` picks a label, and the listener for that label finishes the job.

In [4]:
class SupportFlow(Flow[Support]):  # Flow[Support]: the state is a Support
    @start()  # the first step: runs when the flow is kicked off
    def triage(self):
        # Model call: read the message and return an intent and an order ID (structured output, as in notebook 01).
        t = llm.call([{"role": "system", "content": TRIAGE_PROMPT}, {"role": "user", "content": self.state.message}],
                     response_model=Triage)
        self.state.intent, self.state.order_id = t.intent, t.order_id

    @router(triage)  # runs after triage and returns a label instead of a result
    def route(self):
        # The decision lives in code: out-of-scope messages get the label "decline", everything else "resolve".
        return "decline" if self.state.intent == "other" else "resolve"

    @listen("decline")  # runs only when route returned "decline"
    def send_decline(self):  # no model call at all
        self.state.reply = "Sorry, I can only help with orders, deliveries, refunds and complaints."

    @listen("resolve")  # runs only when route returned "resolve"
    async def run_specialist(self):
        # Agent call: the specialist chooses its own tools and writes the reply.
        self.state.reply = (await specialist.kickoff_async(self.state.message)).raw

Now run the flow on two messages, one out of scope and one about an order. For each you see what triage decided, which tools ran, and the reply.

In [5]:
for message in ("Can you write me a poem about cats?", "Where is order A9999?"):
    shipfast.reset()  # clear the tool-call log, so only this message shows up
    flow = SupportFlow(suppress_flow_events=True)  # no console panel for every step
    await flow.kickoff_async(inputs={"message": message})  # inputs fill the state's fields
    s = flow.state  # the final state, after every step has run
    print("message:", message)
    print("  intent:  ", s.intent)
    print("  order_id:", s.order_id)
    print("  tools:   ", ", ".join(shipfast.tool_calls()) or "none")
    print("  reply:   ", s.reply.strip()[:150])
    print()

message: Can you write me a poem about cats?
  intent:   other
  order_id: None
  tools:    none
  reply:    Sorry, I can only help with orders, deliveries, refunds and complaints.



message: Where is order A9999?
  intent:   order_status
  order_id: A9999
  tools:    lookup_order
  reply:    I couldn't find any order with the ID A9999. Could you please double-check the order ID or provide additional details?



The poem never reached the specialist: a router in code handled it, cheaply and predictably. The order
question went to the agent, which chose its own tools. That mix, **a workflow outside and an agent
inside**, is what CrewAI recommends: start with a Flow for structure, and use agents or crews inside the
steps that need autonomy.

Three practical notes. `suppress_flow_events=True` stops CrewAI printing a panel for every step of the
flow. A listener cannot share its name with the label it listens to: CrewAI rejects
`@listen("decline") def decline` as a possible infinite loop, so the handlers here are `send_decline` and
`run_specialist`. And `kickoff_async` again, because a notebook already runs an event loop; the
`run_specialist` step is `async` for the same reason, since it awaits the agent.

## The research agent as a flow

The research agent (`contracts/agent_contracts/research/SPEC.md`) must plan, research with at most 5 tool
calls, draft a cited answer, critique it, and revise at most twice. Most of that is a fixed sequence, so it
is a flow. Only the research step is open-ended, so that step is an agent. It lives in
`crew_agents/research_flow.py`, with the same instructions as the LangGraph build:

In [6]:
# The docstring of crew_agents/research_flow.py, shown because it draws the flow this notebook explains.
import crew_agents.research_flow as rf

print(rf.__doc__)

The research agent as a CrewAI Flow (research SPEC.md section 1), taught in notebook 05.

Workflow outside, agent inside, as in lg_agents/research_flow.py and with the same instructions: fixed
steps (plan, draft, critique, revise) and a tool-using CrewAI agent for the research step. The revision
limit is enforced in code by the router, never by the prompt; the tool budget (5) by the shared tools.

    plan -> research -> draft -> critique (router) --"done"--> END
                                    ^        \--"needs_revision"--> revise --+
                                    +---------------------------------------+
    The router skips the critique and returns "done" once 2 revisions have run.



## Running it

Question Q01 asks how many employees Veltara Aquatech has. The fixtures contain the current figure (350,
in a 2026 article) and a stale one (200, in a 2023 article), and an employees search returns both.

In [7]:
import time

from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report

questions = {q["id"]: q for q in load_questions()}
q = questions["Q01_headcount"]
research.reset()  # clear the tool-call log and budget

# Run the whole flow on the question; the flow keeps its results in flow.state.
flow = rf.ResearchFlow(llm)
start_time = time.perf_counter()
await flow.kickoff_async(inputs={"question_id": q["id"], "question": q["question"]})
print(f"ran in {time.perf_counter() - start_time:.0f}s")

ran in 58s


Now read the flow's final state, one labelled line per field, and score it with the shared scorer, a plain text check against the question's expected facts.

In [8]:
state = flow.state
score = score_report(rf.to_report(state), q)

print("plan:")
for step in state.plan:
    print("  -", step)
print("tools:     ", ", ".join(research.tool_calls()) or "none")
print("answer:    ", state.answer)
print("citations: ", ", ".join(state.citations) or "none")
print("critiques: ", ", ".join(c["verdict"] for c in state.critiques) or "none")
print("revisions: ", state.revisions)
print("passed:    ", score["passed"])
for name, value in score["scores"].items():
    print(f"  {name + ':':20}", value)
print("problems:  ", score["problems"] or "none")

plan:
  - Search for Veltara Aquatech's official website to find any publicly available information about their employee count.
  - Look for recent press releases, investor reports, or company profiles on platforms like LinkedIn or Crunchbase that may mention the number of employees.
  - Check if the company has disclosed its employee count in annual reports or through regulatory filings (e.g., SEC filings for U.S.-based companies).
  - If the information is not publicly available, consider reaching out to the company through their contact form or customer service for an official response.
tools:      web_search
answer:     Veltara Aquatech currently has 350 employees, according to the most recent report published on 2026-03-18 [F02].
citations:  F02
critiques:  pass
revisions:  0
passed:     True
  fact_recall:         1.0
  facts_missing:       []
  contradiction_free:  True
  citations_valid:     True
problems:   none


When we ran it, the flow answered with the current figure, 350, cited the 2026 article, and the critic
passed it at the first attempt, so the loop never ran. The plan is generic (it suggests LinkedIn,
regulatory filings and emailing the company, none of which the tools can do); it guides the research
agent but nothing checks it.

The LangGraph tutorial ran the same question with the same instructions, and it passed there too. It
does not always: on another run there, the draft added that 350 "supersedes" the older 200. The critic
passed that, but the shared scorer failed it for mentioning the stale figure. A model checking a model misses things a plain text check does not, which is why both
agents are scored in code.

## The loop, and its limit

The loop is two decorators. `critique` is a router that runs after `draft` **or** after `revise`; it
returns `"needs_revision"`, which triggers `revise`, or `"done"`, which nothing listens to, so the flow
ends. The limit sits at the top of the router, in code:

```python
@router(or_(draft, "revise"))  # "revise" here is the method below finishing
def critique(self):
    # A router's return value is a label: a method decorated @listen("<label>") runs next.
    if self.state.revisions >= MAX_REVISIONS:  # the limit, enforced in code: no third critique
        return "done"  # nothing listens for "done", so the flow ends here
    # One structured model call with the critic's prompt (not a separate agent): a verdict and issues.
    c = critique_draft(self.llm, self.state.question, self.state.findings, self.state.answer, self.state.citations)
    self.state.critiques.append(c.model_dump())  # kept in the flow's state, for the report
    return "needs_revision" if c.verdict == "revise" else "done"  # "needs_revision" triggers revise()
```

The critic is not useless: given a draft that relies on the stale figure outright, with the real
findings from the run above, it sends it back:

In [10]:
# A draft that relies on the stale figure (200) outright, checked against the real findings from the run above.
critique = rf.critique_draft(llm, q["question"], state.findings, "Veltara Aquatech has 200 employees [F03].", ["F03"])

print("verdict:", critique.verdict)
print("issues:")
for issue in critique.issues:
    print("  -", issue)

verdict: revise
issues:
  - The draft answer cites F03, which reports 200 employees, but the most recent finding (F02) from 2026-03-18 states that Veltara Aquatech now has 350 employees. The draft answer uses outdated information.
  - The question asks for the current number of employees, and the draft answer does not provide the most recent figure as per the findings.


To see the limit hold, replace the critic with one that is never satisfied, a standard testing trick,
and run another question. However often the critic says "revise", the flow stops after the second
revision without asking a third time.

In [11]:
from agent_contracts.research.schema import Critique

# A stand-in critic for the test: it counts how often it is asked and always says "revise".
real_critic = rf.critique_draft
critic_calls = []


def never_satisfied(*args, **kwargs):
    critic_calls.append(1)
    return Critique(verdict="revise", issues=["(test) always revise"])

Now swap the stand-in in, run the flow on another question, and put the real critic back afterwards (the `finally` makes sure that happens even if the run fails).

In [12]:
rf.critique_draft = never_satisfied
try:
    q4 = questions["Q04_series_b"]
    research.reset()
    flow = rf.ResearchFlow(llm)
    await flow.kickoff_async(inputs={"question_id": q4["id"], "question": q4["question"]})
finally:
    rf.critique_draft = real_critic  # always restore the real critic

Count how often the critic was asked and what the loop looked like. The scorer checks that the verdicts and revisions form one of the allowed shapes.

In [13]:
report = rf.to_report(flow.state)
loop_problems = [p for p in score_report(report, q4)["problems"] if "loop" in p]

print("critic calls:   ", len(critic_calls))
print("verdicts:       ", ", ".join(c.verdict for c in report.critiques))
print("revisions:      ", report.revisions)
print("loop problems:  ", loop_problems or "none")

critic calls:    2
verdicts:        revise, revise
revisions:       2
loop problems:   none


So a run can only end in the three shapes the spec allows: `[pass]`, `[revise, pass]`, or
`[revise, revise]` after two revisions. `score_report` checks that shape for every report.

## Workflow or agent?

| | Workflow (`Flow`) | Agent (`Agent`, or a `Crew` of agents) |
|---|---|---|
| Who decides the next step | your code | the model |
| Predictability | high: the same path every time | lower: it can skip or repeat steps |
| Good for | known procedures, limits that must hold, cheap routing | open-ended investigation |
| In this tutorial | ShipFast routing, the research loop | ShipFast's specialist, the research step |

A `Crew` is CrewAI's other way to combine work: several agents and tasks run by a **process**, either
`sequential` (tasks in order) or `hierarchical` (a manager agent plans and delegates). That is
orchestration by agents rather than by code; Flows are the code-first option, and the two combine.

## How the others do this

- **LangGraph** draws the same thing as a `StateGraph`: a typed state, nodes, and conditional edges whose
  routing functions return the next node's name, the counterpart of a CrewAI router's label.
- **Strands** builds multi-step systems from agents: `GraphBuilder` connects agents as nodes with
  conditional edges, and `Swarm` lets agents hand off to each other. Plain code steps are less central
  than in a Flow or a `StateGraph`.

## Takeaways

- A Flow is a class: `@start`, `@listen` and `@router` wire methods together, and `self.state` carries the
  data between them.
- Use code for the steps you know and an agent for the steps you do not; mix them freely.
- Put limits in the routers. A loop bounded in code cannot run away.
- Next: **06 · Guardrails and approval**.